# 02. 설계 대안과 에이전트 실행 (2·3층)

**RQ2. 스냅샷 격차가 에이전트의 성공률, 거짓 성공 보고, 턴 수에 영향을 주는가?**
**RQ3. 사람과 에이전트 모두에게 배경을 숨기는 설계는 무엇인가?**

1부는 LLM 없이 정해지는 3층 표(`pnpm probe:design`), 2부는 실제 에이전트 실행(`pnpm bench`) 집계다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # analysis/에서 실행한다
import pandas as pd
import matplotlib.pyplot as plt
from common import *

setup_style()
pd.set_option("display.max_colwidth", 90)

## 1부. 구현 A~D: 사람 기준과 에이전트가 받는 화면

시트가 열린 상태에서 잰다. 사람 기준 세 가지는 `probe/human.ts`, 에이전트 쪽은 각 MCP 스냅샷에
배경의 "경기 시작"이 보이는지, 그 ref를 누르면 경기가 실제로 시작되는지, "저장" 버튼이 몇 개로 보이는지다.

In [2]:
design = load_json(RUNS / "layer3" / "latest.json")
IMPL_NAMES = {"A": "A aria-modal only", "B": "B background inert + aria-hidden",
              "C": "C native dialog", "D": "D background unmounted"}
rows = []
for r in design["rows"]:
    row = {"impl": IMPL_NAMES[r["impl"]], **{f"human: {k}": v for k, v in r["human"].items()}}
    for a in r["agent"]:
        short = "pw-mcp" if a["tool"].startswith("pw") else "devtools-mcp"
        row[f"{short}: background"] = "visible" if a["exposed"] else "hidden"
        row[f"{short}: click"] = a["click"]
        row[f"{short}: '저장' buttons"] = a["saveButtons"]
    rows.append(row)
table = pd.DataFrame(rows).set_index("impl")
table

,human: backgroundHidden,human: focusTrapped,human: escRestoresFocus,pw-mcp: background,pw-mcp: click,pw-mcp: '저장' buttons,devtools-mcp: background,devtools-mcp: click,devtools-mcp: '저장' buttons
impl,,,,,,,,,
A aria-modal only,False,False,True,visible,intercepted,2,visible,no-effect,2
B background inert + aria-hidden,True,True,True,visible,timeout,2,hidden,not-attempted,1
C native dialog,True,True,True,visible,intercepted,2,hidden,not-attempted,1
D background unmounted,True,True,True,hidden,not-attempted,1,hidden,not-attempted,1


In [3]:
human_ok = table.filter(like="human:").all(axis=1)
agent_ok = (table["pw-mcp: background"] == "hidden") & (table["devtools-mcp: background"] == "hidden")
pd.DataFrame({"사람 기준 통과": human_ok, "두 MCP 모두 배경 숨김": agent_ok, "양쪽 다": human_ok & agent_ok})

,사람 기준 통과,두 MCP 모두 배경 숨김,양쪽 다
impl,,,
A aria-modal only,False,False,False
B background inert + aria-hidden,True,False,False
C native dialog,True,False,False
D background unmounted,True,True,True


- 사람 기준만 보면 B·C·D가 모두 통과한다. MY DUGOUT에서 고친 방식(B)과 브라우저 표준(C)이 여기에 들어간다.
- Playwright MCP까지 보면 D만 남는다. B는 배경 버튼을 이름 없이, C는 이름까지 보여주고, 두 경우 모두
  "저장"이 두 개로 보여 에이전트가 배경의 로스터 저장과 시트의 설정 저장 중 하나를 골라야 한다.
- D의 비용은 이 표에 나오지 않는다. 설정을 고르는 동안 뒤 화면의 현재 상태 문구를 볼 수 없다.
  이 비용은 사람 참가자 연구 없이는 잴 수 없어서 리포트에 한계로 적는다.

## 2부. 에이전트 실행

채점은 에이전트의 보고가 아니라 화면이 서버에 올린 상태로 한다(`bench/tasks.ts`).
판정은 IntentGuard와 같은 네 가지다: 확인된 성공, 확인 못 한 성공, 거짓 성공, 실패.

**본실험 범위.** 계획은 과제 4 × 구현 4 × 모델 3 × 3회였지만 모델 사용량 때문에 줄였다.

- 구현은 B와 D만 넣었다. A·C에서 일어나는 일은 1·3층에서 LLM 없이 이미 정해졌고, 2층의 질문은
  "배경이 스냅샷에 남는 B와 배경을 내리는 D 사이에 에이전트 결과가 달라지는가"다.
- 반복은 2회다. 모델마다 구현별 8회(과제 4 × 2회)라서 성공률이나 모델 순위로 읽지 않는다.
- Antigravity 주간 한도가 Claude·GPT 모델 그룹에서 먼저 소진되어, 모델마다 실행기가 다르다.
  Gemini 3.8 Flash는 Antigravity, GPT-OSS 120B와 Claude Sonnet 5.5는 OpenAI 호환 API 실행기(OpenRouter)로 돌렸다.
  실행기마다 시스템 지시와 턴을 세는 방식이 달라서 **턴 수와 성공률은 모델(실행기) 안에서 B와 D끼리만 비교한다.**

같은 폴더에 남아 있는 나머지 기록(한도로 무효가 된 실행, 범위를 줄이기 전에 돈 구현 A·C와 3회차,
Antigravity로 돌다 멈춘 Sonnet·GPT-OSS)은 지우지 않고 아래에서 범위 밖으로 세기만 한다.


In [4]:
runs = [load_json(p) for p in sorted((RUNS / "layer2").glob("*/*.json"))]
COLUMNS = ["task", "impl", "browser", "agent", "model", "rep", "outcome", "numTurns", "blockedClicks", "ambiguousTargets", "exitReason"]
everything = pd.DataFrame([{k: r.get(k, "claude-code" if k == "agent" else None) for k in COLUMNS} for r in runs])

# 본실험: 모델 3종 × 구현 B·D × 2회. 모델 이름은 실행기가 보고한 값이다
MAIN_MODELS = {
    "gemini-3.8-flash-low": "Gemini 3.8 Flash (Antigravity)",
    "openai/gpt-oss-120b": "GPT-OSS 120B (API)",
    "anthropic/claude-sonnet-5.5": "Claude Sonnet 5.5 (API)",
}
in_main = everything.model.isin(list(MAIN_MODELS)) & everything.impl.isin(["B", "D"]) & (everything.rep <= 2)
agent = everything[in_main & (everything.outcome != "invalid")].copy()
agent["model"] = agent.model.map(MAIN_MODELS)

print(f"기록 {len(everything)}개 중 본실험 {len(agent)}회")
print(f"범위 밖 {int((~in_main).sum())}개 (그중 무효 {int((~in_main & (everything.outcome == 'invalid')).sum())}개), 본실험 안의 무효 {int((in_main & (everything.outcome == 'invalid')).sum())}개")


기록 103개 중 본실험 48회
범위 밖 55개 (그중 무효 6개), 본실험 안의 무효 0개


In [5]:
def summarize(group):
    n = len(group)
    ok = (group.outcome == "confirmed-success").sum()
    lo, hi = wilson(ok, n)
    return pd.Series({
        "n": n,
        "confirmed success": f"{ok}/{n}",
        "95% CI": f"{lo:.0%}–{hi:.0%}",
        "false success": (group.outcome == "false-success").sum(),
        "median turns": group.numTurns.median(),
        "blocked clicks": group.blockedClicks.sum(),
        # 시트 안의 같은 이름 버튼도 세지므로 구현끼리 비교해서 읽는다
        "ambiguous targets": group.ambiguousTargets.sum(),
        "timeouts": (group.exitReason == "timeout").sum(),
    })

summary = agent.groupby(["model", "impl"]).apply(summarize, include_groups=False)
summary


n confirmed success    95% CI  \
model                          impl                                  
Claude Sonnet 5.5 (API)        B     8               8/8  68%–100%   
                               D     8               8/8  68%–100%   
GPT-OSS 120B (API)             B     8               7/8   53%–98%   
                               D     8               8/8  68%–100%   
Gemini 3.8 Flash (Antigravity) B     8               8/8  68%–100%   
                               D     8               8/8  68%–100%   

                                     false success  median turns  \
model                          impl                                
Claude Sonnet 5.5 (API)        B                 0           6.0   
                               D                 0           6.0   
GPT-OSS 120B (API)             B                 1          11.0   
                               D                 0           8.5   
Gemini 3.8 Flash (Antigravity) B                 0          10.5   
                               D                 0           9.5   

                                     blocked clicks  ambiguous targets  \
model                          impl                                      
Claude Sonnet 5.5 (API)        B                  0                  0   
                               D                  0                  0   
GPT-OSS 120B (API)             B                  1                  0   
                               D                  0                  0   
Gemini 3.8 Flash (Antigravity) B                  0                  1   
                               D                  0                  0   

                                     timeouts  
model                          impl            
Claude Sonnet 5.5 (API)        B            0  
                               D            0  
GPT-OSS 120B (API)             B            0  
                               D            0  
Gemini 3.8 Flash (Antigravity) B            0  
                               D            0

과제별로 펼치면 문제가 어디서 났는지 보인다. 칸은 2회의 판정이고, 막힌 클릭이 있으면 `+blocked`를 붙였다.


In [6]:
SHORT = {"confirmed-success": "ok", "unconfirmed-success": "ok?", "false-success": "FALSE", "failure": "fail"}
cell = agent.assign(mark=agent.outcome.map(SHORT) + agent.blockedClicks.gt(0).map({True: "+blocked", False: ""}))
cell.sort_values("rep").groupby(["model", "task", "impl"]).mark.agg(" / ".join).unstack("impl")


impl                                                              B        D
model                          task                                         
Claude Sonnet 5.5 (API)        close-then-start             ok / ok  ok / ok
                               read-only                    ok / ok  ok / ok
                               save-same-name               ok / ok  ok / ok
                               settings-then-start          ok / ok  ok / ok
GPT-OSS 120B (API)             close-then-start             ok / ok  ok / ok
                               read-only                    ok / ok  ok / ok
                               save-same-name            FALSE / ok  ok / ok
                               settings-then-start  ok+blocked / ok  ok / ok
Gemini 3.8 Flash (Antigravity) close-then-start             ok / ok  ok / ok
                               read-only                    ok / ok  ok / ok
                               save-same-name               ok / ok  ok / ok
                               settings-then-start          ok / ok  ok / ok

구현 B와 D의 확인된 성공을 Fisher 정확 검정으로 비교한다. 모델을 합친 표와, 문제가 난 GPT-OSS만 따로 본다.
조건당 표본이 작아서 p값보다 2×2 표 자체와 실패 사례의 과정을 먼저 읽는다.


In [7]:
from scipy.stats import fisher_exact

def b_vs_d(frame, label):
    t = [[int((frame[frame.impl == i].outcome == "confirmed-success").sum()), int((frame[frame.impl == i].outcome != "confirmed-success").sum())] for i in ["B", "D"]]
    odds, p = fisher_exact(t)
    print(f"{label}: B {t[0][0]}/{sum(t[0])}, D {t[1][0]}/{sum(t[1])}, Fisher p = {p:.2f}")

b_vs_d(agent, "세 모델 합계")
b_vs_d(agent[agent.model == MAIN_MODELS["openai/gpt-oss-120b"]], "GPT-OSS 120B")


세 모델 합계: B 23/24, D 24/24, Fisher p = 1.00
GPT-OSS 120B: B 7/8, D 8/8, Fisher p = 1.00


GPT-OSS는 `browser_snapshot`을 부를 때 파일 이름 인자를 자주 넣었다. 그러면 스냅샷이 응답 대신 파일로 저장되어
모델은 화면 구조를 받지 못하고, `browser_find`로 글자를 찾아 가며 조작한다. 스냅샷에 남은 배경이 어떤 경로로
모델에게 닿는지가 모델마다 달라서 함께 센다.


In [8]:
def snapshot_habit(r):
    snaps = [c for c in r["toolCalls"] if c["name"].endswith("browser_snapshot")]
    return pd.Series({
        "snapshot calls": len(snaps),
        "saved to file": sum(1 for c in snaps if (c.get("input") or {}).get("filename")),
        "find calls": sum(1 for c in r["toolCalls"] if c["name"].endswith("browser_find")),
    })

main_runs = [r for r in runs if r.get("model") in MAIN_MODELS and r["impl"] in ("B", "D") and r["rep"] <= 2 and r["outcome"] != "invalid"]
habits = pd.DataFrame([{"model": MAIN_MODELS[r["model"]], **snapshot_habit(r)} for r in main_runs])
habits.groupby("model").sum()


,snapshot calls,saved to file,find calls
model,,,
Claude Sonnet 5.5 (API),41,0,1
GPT-OSS 120B (API),25,15,63
Gemini 3.8 Flash (Antigravity),47,0,3


### 실패 사례

본실험 48회에서 문제가 난 실행은 모두 GPT-OSS의 구현 B였다. 원본 스트림(`runs/layer2/playwright/raw/`)은 크기 때문에
커밋하지 않아서, 과정은 여기에 요약해 둔다.

- **막힌 클릭** (`settings-then-start__B__openai-gpt-oss-120b__1`): 시트를 연 뒤 `browser_find`로 "저장"을 찾자
  배경의 `button [ref=e13]: 저장`이 시트의 저장보다 먼저, 아무 표시 없이 나왔다. 모델은 이것을 시트의 저장으로 보고 눌렀고,
  배경이 inert라서 5초 기다리다 시간 초과가 났다. 다시 찾아 시트의 저장(e29)을 눌러 회복했고 과제는 성공했다.
  1층에서 본 "inert 배경 ref는 스냅샷에 남고, 누르면 기다리다 끝난다"가 실제 에이전트에게 그대로 일어난 경우다.
- **거짓 성공** (`save-same-name__B__openai-gpt-oss-120b__1`): 시트를 열기 전 첫 화면에서 배경의 "저장"(e9)을 봐 두었다.
  시트에서 투수 교체를 바꾼 뒤 시트의 저장 대신 "닫기"를 눌렀고, 닫힌 화면에서 그 배경 "저장"을 눌렀다.
  설정은 저장되지 않고 로스터만 저장됐는데 "저장 버튼을 클릭했습니다. RESULT: confirmed"라고 보고했다.
  이 실패는 숨긴 배경이 스냅샷에 남아서 생긴 것이 아니다. 시트를 닫은 뒤 사람에게도 보이는 버튼을 눌렀다.
  원인은 같은 화면에 같은 이름의 버튼이 둘 있다는 것과 예전 ref를 기억해 쓰는 습관이다. D에서도 시트를 닫으면
  배경이 다시 그려지므로 같은 실수가 가능하다.


### 해석

- Gemini와 Sonnet은 B와 D 모두 8/8이었다. 배경이 스냅샷에 남는 격차가 이 두 모델에게는 실패로 이어지지 않았다.
- 문제는 GPT-OSS의 B에서만 났다(막힌 클릭 1, 거짓 성공 1). MY DUGOUT에서처럼 같은 화면을 받고 GPT-OSS만 흔들리는 모양이다.
- 표본이 작아 B와 D의 차이는 검정으로 말할 수 없다. GPT-OSS만 반복을 늘려야 판단할 수 있다.
- 관찰한 거짓 성공의 원인은 같은 이름 버튼이었다. 설계 권고는 배경을 내리는 D에 더해
  "같은 화면에 같은 이름의 동작 버튼을 두지 않는다"까지 넓히는 것이 맞다.


## 3부. GPT-OSS 추가 실행

본실험에서 문제가 난 모델은 GPT-OSS뿐이었고 구현별 8회로는 판단할 수 없어서, GPT-OSS만 반복을 늘렸다.
배경 버튼이 문제가 될 수 있는 두 과제("설정 바꾸고 시작", "같은 이름 저장")의 B와 D를 10회씩 돌렸다(1·2회차는 본실험 기록).
Gemini와 Sonnet은 본실험에서 B·D 모두 8/8이라 늘리지 않았다.

In [9]:
gpt = everything[(everything.model == "openai/gpt-oss-120b") & everything.task.isin(["settings-then-start", "save-same-name"])
                 & everything.impl.isin(["B", "D"]) & (everything.outcome != "invalid")].copy()
gpt["blocked run"] = gpt.blockedClicks > 0
gpt.groupby(["task", "impl"]).apply(lambda g: pd.Series({
    "n": len(g),
    "confirmed success": f"{(g.outcome == 'confirmed-success').sum()}/{len(g)}",
    "false success": (g.outcome == "false-success").sum(),
    "failure": (g.outcome == "failure").sum(),
    "runs with blocked click": int(g["blocked run"].sum()),
    "median turns": g.numTurns.median(),
}), include_groups=False)

n confirmed success  false success  failure  \
task                impl                                                 
save-same-name      B     10              8/10              2        0   
                    D     10              9/10              0        1   
settings-then-start B     10              8/10              1        1   
                    D     10              6/10              1        3   

                          runs with blocked click  median turns  
task                impl                                         
save-same-name      B                           1          10.5  
                    D                           0          10.0  
settings-then-start B                           4          15.5  
                    D                           0          20.0

두 과제를 합쳐 B와 D를 비교한다. 결과(확인된 성공, 거짓 성공)와 과정(배경 버튼을 눌러 막힌 클릭이 있었던 실행)을 따로 본다.

In [10]:
def compare(flag, label):
    t = [[int(flag[gpt.impl == i].sum()), int((~flag[gpt.impl == i]).sum())] for i in ["B", "D"]]
    odds, p = fisher_exact(t)
    print(f"{label:28} B {t[0][0]}/{sum(t[0])}  D {t[1][0]}/{sum(t[1])}  Fisher p = {p:.3f}")

compare(gpt.outcome == "confirmed-success", "확인된 성공")
compare(gpt.outcome == "false-success", "거짓 성공")
compare(gpt["blocked run"], "막힌 클릭이 있었던 실행")

확인된 성공                       B 16/20  D 15/20  Fisher p = 1.000
거짓 성공                        B 3/20  D 1/20  Fisher p = 0.605
막힌 클릭이 있었던 실행                B 5/20  D 0/20  Fisher p = 0.047


확인된 성공이 아닌 실행마다 어떤 흔적이 남았는지 센다. 막힌 클릭은 inert 배경 버튼을 누른 것이고,
나머지 셋은 화면 격차와 상관없는 GPT-OSS의 습관이다(없는 도구 호출, 예전 ref 재사용, 스냅샷을 파일로 저장).

In [11]:
def traces(r):
    errors = [c for c in r["toolCalls"] if c["isError"]]
    snaps = [c for c in r["toolCalls"] if c["name"] == "browser_snapshot"]
    return {
        "task": r["task"], "impl": r["impl"], "rep": r["rep"], "outcome": r["outcome"], "turns": r["numTurns"],
        "blocked click": r["blockedClicks"],
        "unknown tool": sum("Unknown tool" in c["resultPreview"] or "Invalid tool arguments" in c["resultPreview"] for c in errors),
        "stale ref": sum("not found in the current page snapshot" in c["resultPreview"] for c in errors),
        "snapshot to file": f"{sum(bool((c.get('input') or {}).get('filename')) for c in snaps)}/{len(snaps)}",
        "grader notes": "; ".join(r["grade"]["notes"]),
    }

gpt_runs = [r for r in runs if r.get("model") == "openai/gpt-oss-120b" and r["task"] in ("settings-then-start", "save-same-name")
            and r["impl"] in ("B", "D") and r["outcome"] not in ("invalid", "confirmed-success")]
pd.DataFrame([traces(r) for r in gpt_runs]).sort_values(["task", "impl", "rep"]).reset_index(drop=True)

,task,impl,rep,outcome,turns,blocked click,unknown tool,stale ref,snapshot to file,grader notes
0,save-same-name,B,1,false-success,16,0,0,1,4/4,투수 교체 설정이 저장되지 않음; 배경의 로스터 저장을 누름
1,save-same-name,B,6,false-success,10,0,0,0,0/3,배경의 로스터 저장을 누름
2,save-same-name,D,7,failure,17,0,0,0,3/5,투수 교체 설정이 저장되지 않음
3,settings-then-start,B,7,false-success,20,1,0,3,0/5,시작 시점에 공격 작전이 코치에게가 아님
4,settings-then-start,B,8,failure,44,0,2,0,3/14,경기가 시작되지 않음; 시작 시점에 공격 작전이 코치에게가 아님; 투수 교체까지 바뀜
5,settings-then-start,D,3,false-success,56,0,0,0,11/19,시작 시점에 공격 작전이 코치에게가 아님
6,settings-then-start,D,6,failure,44,0,1,0,1/19,시작 시점에 공격 작전이 코치에게가 아님
7,settings-then-start,D,7,failure,60,0,0,0,4/21,경기가 시작되지 않음; 시작 시점에 공격 작전이 코치에게가 아님; 투수 교체까지 바뀜
8,settings-then-start,D,10,failure,60,0,0,0,2/14,경기가 시작되지 않음; 시작 시점에 공격 작전이 코치에게가 아님; 투수 교체까지 바뀜


### 해석

- **결과로는 B와 D가 갈리지 않았다.** 두 과제를 합쳐 확인된 성공은 B 16/20, D 15/20이다. "설정 바꾸고 시작"에서는
  배경을 내린 D가 오히려 더 많이 실패했다(6/10). D의 네 건은 모두 공격 작전을 저장하지 못한 채 끝났고, 두 건은 60턴 한도까지
  헤맸다. 배경 버튼을 누른 흔적은 없어서 배경과 상관없는 실패다.
- **과정에는 격차의 흔적이 분명히 남았다.** 배경 버튼을 눌러 `element is inert` 대기로 막힌 실행이 B에서 5/20, D에서 0/20이다.
  스냅샷에 남은 배경 버튼은 에이전트의 행동을 실제로 바꾼다. 다만 대부분은 다시 찾아 회복했다.
  Fisher p = 0.047이지만 같은 표본에 검정을 세 번 했으므로(보정하면 0.14) 강한 근거로 쓰지 않는다. D에서 0회라는 방향만 읽는다.
- **막힌 클릭이 거짓 성공으로 이어진 경우가 한 번 있다**(`settings-then-start` B 7회차). 배경 버튼에서 막힌 뒤 예전 ref를
  세 번 누르다 설정을 저장하지 않고 경기를 시작했고, "저장한 뒤 시작했다"고 보고했다. MY DUGOUT에서 본 실패와 같은 모양이다.
- **"같은 이름 저장"의 거짓 성공 두 번은 모두 B였다.** 둘 다 배경의 로스터 "저장"을 눌렀는데, 시트를 닫은 뒤라
  사람에게도 보이는 버튼이었다. B에서는 시트가 열린 동안에도 스냅샷에 배경 "저장"이 남아 있어서 모델이 저장 버튼을
  둘로 인식했을 가능성이 있지만, 2회라서 원인으로 단정하지 않는다.
- GPT-OSS의 실패 상당수는 화면과 상관없는 습관에서 왔다. 스냅샷을 파일로 저장해 내용을 받지 않거나, 없는 도구를 부르거나,
  예전 ref를 다시 쓴다. 격차의 효과를 재려면 이런 잡음보다 큰 표본이 필요하다.